## Step 1: Load the snapshot

In [8]:
import pandas as pd

# Load your combined CSV
df = pd.read_csv("../data/raw/crypto_market_snapshot_2026-02-01.csv")

# Quick view
print(df.shape)
df.head()


(732, 18)


,timestamp,price,market_cap,volume,coin,symbol,name,market_cap_rank,circulating_supply,total_supply,max_supply,ath,atl,price_change_24h,price_change_percentage_24h,categories,bullish_votes_pct,bearish_votes_pct
0,2025-02-02 00:00:00,100674.787625,1.996424e+12,2.282778e+10,bitcoin,btc,Bitcoin,1,19982656.0,19982656.0,21000000.0,126080.0,67.81,-5555.682663,-6.62719,"Smart Contract Platform, Layer 1 (L1), FTX Hol...",53.62,46.38
1,2025-02-03 00:00:00,97568.316530,1.933691e+12,5.978423e+10,bitcoin,btc,Bitcoin,1,19982656.0,19982656.0,21000000.0,126080.0,67.81,-5555.682663,-6.62719,"Smart Contract Platform, Layer 1 (L1), FTX Hol...",53.62,46.38
2,2025-02-04 00:00:00,101466.860666,2.011121e+12,1.221640e+11,bitcoin,btc,Bitcoin,1,19982656.0,19982656.0,21000000.0,126080.0,67.81,-5555.682663,-6.62719,"Smart Contract Platform, Layer 1 (L1), FTX Hol...",53.62,46.38
3,2025-02-05 00:00:00,98118.439217,1.943535e+12,7.319669e+10,bitcoin,btc,Bitcoin,1,19982656.0,19982656.0,21000000.0,126080.0,67.81,-5555.682663,-6.62719,"Smart Contract Platform, Layer 1 (L1), FTX Hol...",53.62,46.38
4,2025-02-06 00:00:00,96582.886829,1.912585e+12,4.884896e+10,bitcoin,btc,Bitcoin,1,19982656.0,19982656.0,21000000.0,126080.0,67.81,-5555.682663,-6.62719,"Smart Contract Platform, Layer 1 (L1), FTX Hol...",53.62,46.38


## Step 2: check the data 

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 732 entries, 0 to 731
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   timestamp                    732 non-null    object 
 1   price                        732 non-null    float64
 2   market_cap                   732 non-null    float64
 3   volume                       732 non-null    float64
 4   coin                         732 non-null    object 
 5   symbol                       732 non-null    object 
 6   name                         732 non-null    object 
 7   market_cap_rank              732 non-null    int64  
 8   circulating_supply           732 non-null    float64
 9   total_supply                 732 non-null    float64
 10  max_supply                   366 non-null    float64
 11  ath                          732 non-null    float64
 12  atl                          732 non-null    float64
 13  price_change_24h    

In [3]:
df.shape


(732, 18)

In [4]:
#check the missing values 
(df.isnull().sum())

timestamp                        0
price                            0
market_cap                       0
volume                           0
coin                             0
symbol                           0
name                             0
market_cap_rank                  0
circulating_supply               0
total_supply                     0
max_supply                     366
ath                              0
atl                              0
price_change_24h                 0
price_change_percentage_24h      0
categories                       0
bullish_votes_pct                0
bearish_votes_pct                0
dtype: int64

In [5]:
df.groupby("coin")["max_supply"].agg(
    non_null_count="count",
    total_rows="size",
    unique_values="nunique",
    value="first"
)

,non_null_count,total_rows,unique_values,value
coin,,,,
bitcoin,366,366,1,21000000.0
ethereum,0,366,0,NaN


* Bitcoin has a defined max_supply = 21,000,000
Ethereum has no fixed max supply which explaines why the max_supply column is null for ETH 
This is a semantic missingness, not a data quality issue. we will address this later in the feature engineering stage by setting a binary flag (1 is max supply exists else 0)

In [6]:
#check duplicates 
(df.duplicated().sum())

np.int64(0)

* i noticed that the features " price_change_24h" "price_change_percentage_24h" have an identical value every day, let's check this further 

In [9]:
df.groupby("coin")[["price_change_24h", "price_change_percentage_24h"]].nunique()


,price_change_24h,price_change_percentage_24h
coin,,
bitcoin,1,1
ethereum,1,1


* The features price_change_24h and price_change_percentage_24h provided by the CoinGecko coin info endpoint are snapshot values and remain constant across the historical dataset.

* To avoid data leakage and time misalignment, a time-consistent 24h price change will be computed later directly from historical prices as part of the feature engineering phase.

In [12]:
#data type check 
(df.dtypes)

timestamp                       object
price                          float64
market_cap                     float64
volume                         float64
coin                            object
symbol                          object
name                            object
market_cap_rank                  int64
circulating_supply             float64
total_supply                   float64
max_supply                     float64
ath                            float64
atl                            float64
price_change_24h               float64
price_change_percentage_24h    float64
categories                      object
bullish_votes_pct              float64
bearish_votes_pct              float64
dtype: object

* the timestamp feature is in object type we will convert it to datetime in the data preprocessing stage ( that is mandatory )

In [13]:
#identify which columns are likely categorical
(df.nunique())

timestamp                      367
price                          732
market_cap                     732
volume                         732
coin                             2
symbol                           2
name                             2
market_cap_rank                  2
circulating_supply               2
total_supply                     2
max_supply                       1
ath                              2
atl                              2
price_change_24h                 2
price_change_percentage_24h      2
categories                       2
bullish_votes_pct                2
bearish_votes_pct                2
dtype: int64

* i noticed that the "circulating_supply" feature is not historical but rather a snapshot value taken at the time of the data extraction, in the feature engineering stage we will compute a new feature " circulation_supply_history" using the marketcap and price features which will be much better then a static snapshot value 

* redundancy is identified for the features "coin","symbol","name" , in the data preprocessing part we will keep only 1 of them ( probably "coin")

* "categories" ex : Smart Contract Platform, Layer 1 (L1), FTX Holdings Recovery. This tells what the coin is, not how it behaves over time. we will drop it in the data preprocessing stage.

* ath ( all time high ) and atl ( all time low ) have a single unique value per coin captured at the time os the snapshot, they don't vary over time but we will keepthem to extract a new feature "distance from ath" and "distance from atl" in the feature engineering stage.

* the bullish_votes_pct bearish_votes_pct features are also snapshot values we will drop them in the EDA and ML/ deeplearning model but we will uclude them later when it comes to the chatbot development part as they reflect the market sentiment at a given time. 

In [14]:
#check the abnormal values (Price, market cap, volume must be > 0)
numeric_cols = ["price", "market_cap", "volume"]

for col in numeric_cols:
    print(col, (df[col] <= 0).sum())

price 0
market_cap 0
volume 0


## Step 3: Data preprocessing 

In [16]:
# round the values 
df["price"] = df["price"].round(2)
df["market_cap"] = df["market_cap"].round(0)
df["volume"] = df["volume"].round(0)
df["circulating_supply"] = df["circulating_supply"].round(0)
df["total_supply"] = df["total_supply"].round(0)
df["max_supply"] = df["max_supply"].round(0)
df["ath"] = df["ath"].round(2)
df["atl"] = df["atl"].round(2)
df["price_change_24h"] = df["price_change_24h"].round(2)
df["price_change_percentage_24h"] = df["price_change_percentage_24h"].round(2)
df["bullish_votes_pct"] = df["bullish_votes_pct"].round(2)
df["bearish_votes_pct"] = df["bearish_votes_pct"].round(2)

In [19]:
#Feature Selection Drop / Keep

cols_to_drop = [
    "bullish_votes_pct",
    "bearish_votes_pct",
    "categories",
    "symbol",
    "name"
]

df = df.drop(columns=cols_to_drop)


In [21]:
#Convert timestamp to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])


In [22]:
df = df.sort_values(["coin", "timestamp"]).reset_index(drop=True)
df.head()

,timestamp,price,market_cap,volume,coin,market_cap_rank,circulating_supply,total_supply,max_supply,ath,atl,price_change_24h,price_change_percentage_24h
0,2025-02-02,100674.79,1.996424e+12,2.282778e+10,bitcoin,1,19982656.0,19982656.0,21000000.0,126080.0,67.81,-5555.68,-6.63
1,2025-02-03,97568.32,1.933691e+12,5.978423e+10,bitcoin,1,19982656.0,19982656.0,21000000.0,126080.0,67.81,-5555.68,-6.63
2,2025-02-04,101466.86,2.011121e+12,1.221640e+11,bitcoin,1,19982656.0,19982656.0,21000000.0,126080.0,67.81,-5555.68,-6.63
3,2025-02-05,98118.44,1.943535e+12,7.319669e+10,bitcoin,1,19982656.0,19982656.0,21000000.0,126080.0,67.81,-5555.68,-6.63
4,2025-02-06,96582.89,1.912585e+12,4.884896e+10,bitcoin,1,19982656.0,19982656.0,21000000.0,126080.0,67.81,-5555.68,-6.63


In [24]:
df.index.is_monotonic_increasing
df.groupby("coin").size()


coin
bitcoin     366
ethereum    366
dtype: int64

## Step 4: Feature engineering 

* We will create a new feature that measures change in price from one day to the next, providing insights into the coin's daily performance and volatility​​ 
- log returns= ln(Price today/Price yesterday)
- this feature will replace the price_change_24h and price_change_percentage_24h features which are snapshot values and remain constant across the historical dataset.


In [27]:
import numpy as np

df["log_return_1d"] = np.log(df["price"] / df.groupby("coin")["price"].shift(1))


* we will create a new feature to mesure the rolling risk of the coin in a given period (7 days, 14 days) 
So instead of just knowing today s return, we know how unstable or risky the coin is in the last period. 


In [48]:
df["volatility_7d"] = (
    df.groupby("coin")["log_return_1d"]
    .rolling(window=7)
    .std()
    .reset_index(level=0, drop=True)
)

df["volatility_14d"] = (
    df.groupby("coin")["log_return_1d"]
    .rolling(window=14)
    .std()
    .reset_index(level=0, drop=True)
)

* we will create a new feature for the RSI 14 day (relative strength index) which evaluates the overbought or oversold condition of a coin base on its recent price movements

In [29]:
window = 14

delta = df.groupby("coin")["price"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.groupby(df["coin"]).rolling(window).mean().reset_index(level=0, drop=True)
avg_loss = loss.groupby(df["coin"]).rolling(window).mean().reset_index(level=0, drop=True)

rs = avg_gain / avg_loss
df["rsi_14"] = 100 - (100 / (1 + rs))


* we will create a new feature for the market structure ratio 

- High value → high trading activity
- Low value → illiquid or sleepy market


In [31]:
df["volume_to_marketcap"] = df["volume"] / df["market_cap"]


* we will compute a new feature "circulating_supply_history" using the marketcap and price features which will replace the circulating_supply snapshot.

In [34]:
df["circulating_supply_history"] = df["market_cap"] / df["price"]

In [35]:
df["circulating_supply_history"] = df["circulating_supply_history"].round(0)

In [38]:
df = df.drop(columns=["circulating_supply"])

* we will add add 2 features ATH / ATL distance features : 
- Near ATH → bullish regime
- Near ATL → distress regime

In [39]:
df["distance_from_ath"] = (df["price"] - df["ath"]) / df["ath"]
df["distance_from_atl"] = (df["price"] - df["atl"]) / df["atl"]

* As the max supply feature has a semantic missingness ( it is null for coins that have no max supply like ETH) we will create a binary flag for the max supply feature 1 if there is max supply else 0
- this feature will replace the max supply feature 

In [43]:
#Create a binary flag for the max supply feature 1 if there is max supply else 0
df["has_max_supply"] = df["max_supply"].notna().astype(int)

In [ ]:
cols_to_drop = [
    "total_supply",
    "max_supply",
    "price_change_24h",
    "price_change_percentage_24h",
    "supply_diff"
]

df = df.drop(columns=cols_to_drop)


KeyError: "['total_supply', 'max_supply', 'price_change_24h', 'price_change_percentage_24h', 'supply_diffreturn_1d'] not found in axis"

In [50]:
df = df.drop(columns=["return_1d"])

* Rolling features such as volatility and RSI require a historical window of 7–14 days. The 14 first rows per coin do not have sufficient data to compute these indicators and were therefore removed to ensure consistency across the database.

In [52]:
rolling_features = ["volatility_7d", "volatility_14d", "rsi_14"]
df = df.dropna(subset=rolling_features).reset_index(drop=True)

In [53]:
pd.set_option('display.max_columns', None)
df.head()

,timestamp,price,market_cap,volume,coin,market_cap_rank,ath,atl,log_return_1d,volatility_7d,volatility_14d,rsi_14,volume_to_marketcap,has_max_supply,circulating_supply_history,distance_from_ath,distance_from_atl
0,2025-03-01,84441.90,1.674754e+12,8.069524e+10,bitcoin,1,126080.0,67.81,-0.003160,0.027420,0.022681,19.605990,0.048183,1,19833216.0,-0.330251,1244.272084
1,2025-03-02,86005.26,1.705564e+12,3.063447e+10,bitcoin,1,126080.0,67.81,0.018345,0.029763,0.023796,24.798349,0.017961,1,19830928.0,-0.317852,1267.327090
2,2025-03-03,94261.53,1.868322e+12,6.185911e+10,bitcoin,1,126080.0,67.81,0.091665,0.050924,0.035796,46.830387,0.033109,1,19820617.0,-0.252367,1389.083026
3,2025-03-04,86124.71,1.708199e+12,6.871536e+10,bitcoin,1,126080.0,67.81,-0.090277,0.058469,0.042980,37.146348,0.040227,1,19834017.0,-0.316904,1269.088630
4,2025-03-05,87310.81,1.731442e+12,6.586336e+10,bitcoin,1,126080.0,67.81,0.013678,0.058173,0.043346,39.356048,0.038040,1,19830782.0,-0.307497,1286.580150
